# Create & Populate QUAM State — OPX1000 (MW-FEM)

Standalone notebook for initial hardware bring-up.

**Steps**:
1. [Preamble](#0-preamble) — switch to the correct Qualibrate project
2. [Create QUAM state](#1-create-quam-state) — allocate wiring and build `state.json` / `wiring.json` *(run once)*
3. [Populate QUAM with initial values](#2-populate-quam-with-initial-values) — write all hardware parameters into the state *(re-run to reset)*
4. [Verify state](#3-verify-state) — sanity-check the saved structure

> **Hardware**: OPX1000 with MW-FEM (slots 1–2) and LF-FEM (slot 3) on controller 1.

## 0. Preamble

Run this cell first every session to select the correct Qualibrate project.

In [ ]:
import sys, os
# Add this notebook's directory to sys.path so the local quam_config package
# is found before any installed version.
_here = os.path.abspath('')
if _here not in sys.path:
    sys.path.insert(0, _here)
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path
from qualibrate_config.core.project.switch import switch_project

config_path = get_qualibrate_config_path()
config = get_qualibrate_config(config_path)
print(f"Current project: {config.project}")

desired_project = "dr3_run9_srf_qubit_1"  # <-- edit if needed
if config.project != desired_project:
    switch_project(config_path, desired_project)
    config = get_qualibrate_config(config_path)
    print(f"Switched to project: {config.project}")
else:
    print(f"Project already set to '{desired_project}'")

print(f"Storage location: {config.storage.location}")

## 1. Create QUAM state

Run **once** to build `state.json` and `wiring.json` in `quam_state/`.
Skip (or re-run to regenerate wiring) if the files already exist.

**Hardware layout**:
- MW-FEM slot 1: readout resonator (port 1, in+out) + qubit XY drive (port 2, out)
- MW-FEM slot 2: cavity drives (f0g1 port 1, Alice port 2, Bob port 3)
- LF-FEM slot 3: flux lines (unused for fixed-frequency transmon)

In [ ]:
import matplotlib.pyplot as plt
from qualang_tools.wirer.wirer.channel_specs import *
from qualang_tools.wirer import Instruments, Connectivity, allocate_wiring, visualize
from quam_builder.builder.qop_connectivity import build_quam_wiring
from quam_builder.builder.superconducting import build_quam
from quam_config import Quam

########################################################################################################################
# %%                                              Define static parameters
########################################################################################################################
host_ip      = "192.168.3.50"    # QOP IP address
port         = None              # QOP port (None = default)
cluster_name = "Cluster_OPX1000" # Name of the OPX1000 cluster

########################################################################################################################
# %%                                      Define the available instrument setup
########################################################################################################################
instruments = Instruments()
instruments.add_mw_fem(controller=1, slots=[1, 2])
instruments.add_lf_fem(controller=1, slots=[3])

########################################################################################################################
# %%                                 Define which qubit IDs are present in the system
########################################################################################################################
qubits = [1]
# qubit_pairs = [(qubits[i], qubits[i + 1]) for i in range(len(qubits) - 1)]

########################################################################################################################
# %%                                 Define any custom/hardcoded channel addresses
########################################################################################################################
# Readout for qubit 1 on MW-FEM slot 1
q1_res_ch   = mw_fem_spec(con=1, slot=1, in_port=1, out_port=1)
# XY drive for qubit 1 on MW-FEM slot 1
q1_drive_ch = mw_fem_spec(con=1, slot=1, in_port=None, out_port=2)

########################################################################################################################
# %%                Allocate the wiring to the connectivity object based on the available instruments
########################################################################################################################
connectivity = Connectivity()
connectivity.add_resonator_line(qubits=qubits[0], constraints=q1_res_ch)
connectivity.add_qubit_drive_lines(qubits=qubits[0], constraints=q1_drive_ch)
# connectivity.add_qubit_flux_lines(qubits=qubits)  # uncomment for tunable qubit
allocate_wiring(connectivity, instruments)

# View wiring schematic
visualize(connectivity.elements, available_channels=instruments.available_channels)
plt.show(block=False)

########################################################################################################################
# %%                                   Build the wiring and QUAM
########################################################################################################################
user_input = input("Do you want to save the updated QUAM? (y/n) ")
if user_input.lower() == "y":
    machine = Quam()
    # Build wiring.json and initialise the QUAM component tree
    build_quam_wiring(connectivity, host_ip, cluster_name, machine)

    # Reload QUAM, build the full object and save state.json
    machine = Quam.load()
    build_quam(machine)
    print("QUAM state created and saved.")
else:
    print("Skipped — no changes saved.")

## 2. Populate QUAM with initial values

Edit the **USER PARAMETERS** section to match your chip specs, then run.

Re-run at any time to reset the state back to the starting-point values.
Calibration nodes overwrite specific fields; this cell resets everything.

In [ ]:
"""
Populate the QUAM state (OPX1000 / MW-FEM) with initial hardware parameters.

Hardware routing (con1):
    FEM 1, port 1  -> readout resonator  (defined in wiring.json)
    FEM 1, port 2  -> qubit XY drive     (defined in wiring.json)
    FEM 2, port 1  -> f0g1 sideband drive
    FEM 2, port 2  -> Alice cavity mode
    FEM 2, port 3  -> Bob cavity mode
"""

########################################################################################################################
# %%  Imports
########################################################################################################################
import json
from pprint import pprint

import numpy as np
from qualang_tools.units import unit
from quam.components.pulses import SquarePulse, DragCosinePulse, DragGaussianPulse
from quam_builder.architecture.superconducting.components.xy_drive import XYDriveMW
from quam_builder.architecture.superconducting.cavity.cavity import Cavity
from quam_builder.architecture.superconducting.cavity.cavity_mode import CavityMode
from quam_builder.builder.superconducting.pulses import add_DragGaussian_pulses
from quam_config import Quam, TemporaryCalibrationData
from quam_builder.architecture.superconducting.qubit_pair import CavityTransmonPair

u = unit(coerce_to_integer=True)

########################################################################################################################
# %%  Helpers
########################################################################################################################
def get_band(freq: float) -> int:
    """Return MW-FEM Nyquist band for a given frequency (Hz).
        Band 1: 50 MHz  -- 5.5 GHz
        Band 2: 4.5 GHz -- 7.5 GHz
        Band 3: 6.5 GHz -- 10.5 GHz
    """
    if 50e6 <= freq < 5.5e9:
        return 1
    elif 4.5e9 <= freq < 7.5e9:
        return 2
    elif 6.5e9 <= freq <= 10.5e9:
        return 3
    else:
        raise ValueError(f"Frequency {freq} Hz is outside MW-FEM range [50 MHz, 10.5 GHz]")


def get_full_scale_power_dBm_and_amplitude(desired_power: float, max_amplitude: float = 0.5) -> tuple[int, float]:
    """Return (full_scale_power_dBm, waveform_amplitude) targeting desired_power (dBm).

    MW-FEM full_scale_power_dbm range: -11 to +16 dBm in 3 dBm steps.
    """
    allowed = [-11, -8, -5, -2, 1, 4, 7, 10, 13, 16]
    resulting = desired_power - 20 * np.log10(max_amplitude)
    if resulting < 0:
        fsp = min(allowed, key=lambda x: abs(x - max(resulting + 3, -11)))
    else:
        fsp = min(allowed, key=lambda x: abs(x - min(resulting + 3, 16)))
    amp = 10 ** ((desired_power - fsp) / 20)
    if not (-11 <= fsp <= 16 and -1 <= amp <= 1):
        raise ValueError(f"Power outside spec: fsp={fsp} dBm, amp={amp:.4f}")
    return fsp, amp


########################################################################################################################
# %%  Load machine
########################################################################################################################
machine = Quam.load()

########################################################################################################################
# %%  USER PARAMETERS -- edit to match chip specs
########################################################################################################################
CAVITY_ID = "c1"

# -- OPX1000 MW-FEM port assignments ----------------------------------------------
#    FEM 1 ports 1-2 are defined in wiring.json; do not change them here.
QUBIT_FEM_ID   = 1   # FEM slot for resonator + qubit XY (from wiring.json)
CAVITY_FEM_ID  = 2   # FEM slot for all cavity-related drives
F0G1_PORT      = 1   # f0g1 sideband drive on CAVITY_FEM_ID
ALICE_PORT     = 2   # Alice cavity mode on CAVITY_FEM_ID
BOB_PORT       = 3   # Bob cavity mode on CAVITY_FEM_ID

# -- Readout resonator -----------------------------------------------------------
rr_freq           = 7.500e9   # Hz  dressed resonator frequency
rr_LO             = 7.400e9   # Hz  upconverter_frequency for FEM 1, port 1
readout_power     = -10        # dBm desired output power

# -- Qubit XY drive --------------------------------------------------------------
xy_freq           = 4.600e9   # Hz  qubit ge transition frequency
xy_LO             = 4.400e9   # Hz  upconverter_frequency for FEM 1, port 2
anharmonicity     = -200e6    # Hz  transmon anharmonicity (negative)
drive_power       = -10        # dBm qubit drive power

# -- Cavity drives (alice + bob share upconverter_frequency) ---------------------
alice_freq        = 6.000e9   # Hz  Alice cavity mode frequency
alice_LO          = 5.900e9   # Hz  upconverter_frequency (shared by alice + bob)
alice_power       = 10         # dBm cavity drive power
bob_freq          = 6.200e9   # Hz  Bob cavity mode frequency
bob_power         = 10         # dBm cavity drive power (same upconverter as alice)

# -- f0g1 sideband drive ---------------------------------------------------------
alice_f0g1_freq                 = 3.25e9   # Hz  initial estimate; refine after node 21
alice_f0g1_LO                   = 3.0e9    # Hz  upconverter_frequency for f0g1 port
alice_f0g1_power                = 0         # dBm initial output power
alice_f0g1_saturation_length_ns = 20000    # ns  long saturation pulse for spectroscopy
alice_f0g1_pi_length_ns         = 1000     # ns  Gaussian pi pulse length
alice_f0g1_sigma_ns             = 200      # ns  Gaussian sigma of f0g1_pi pulse
alice_f0g1_amp                  = 0.4      # V   initial pulse amplitude

# -- Resonator bare frequency and timing -----------------------------------------
rr_freq_bare      = 7.504e9   # Hz  bare frequency (before dispersive shift)
tof_ns            = 224        # ns  time-of-flight (refine with node 01a_time_of_flight)

# -- Qubit reset / timing --------------------------------------------------------
thermalization_time_factor = 5   # x T1 wait time
sigma_time_factor          = 5

# -- EF pi-pulse (initial values; calibrated by nodes 12 / 13) -------------------
ef_x180_amplitude  = 0.1    # V   rough initial amplitude
ef_x180_sigma_ns   = 8      # ns  (references ge x180 sigma via QUAM "#../x180/sigma")

# -- Selective pi-pulse (narrow-bandwidth; calibrated by node 04b_power_rabi) ----
selective_x180_length_ns = 10000  # ns  10 us -> ~100 kHz bandwidth

# -- Pulse defaults --------------------------------------------------------------
readout_length_ns                = 8000
saturation_length_ns             = 20000
x180_length_ns                   = 1000
gaussian_sigma_ns                = x180_length_ns // 5   # 200 ns
drag_alpha                       = 0.0
drag_detuning                    = 0.0
displacement_length_ns           = 1000
displacement_sigma_ns            = displacement_length_ns // 5
displacement_initial_amplitude_V = 0.001  # V (adjust if peak is outside sweep range)

# -- T1 estimates ----------------------------------------------------------------
T1                          = 200e-6   # seconds  transmon
cavity_T1                   = 100e-6   # seconds  cavity modes
resonator_depletion_time_ns = 10000    # ns

########################################################################################################################
# %%  Assertions
########################################################################################################################
assert abs(rr_freq - rr_LO)                  < 400e6, "Resonator IF out of range"
assert abs(xy_freq - xy_LO)                  < 400e6, "XY IF out of range"
assert abs(alice_freq - alice_LO)            < 400e6, "Alice IF out of range"
assert abs(bob_freq   - alice_LO)            < 400e6, "Bob IF out of range (must share upconverter with Alice)"
assert abs(alice_f0g1_freq - alice_f0g1_LO)  < 400e6, "f0g1 IF out of range"
print("All frequency assertions passed.")

########################################################################################################################
# %%  Ensure cavity MW-FEM ports exist on CAVITY_FEM_ID (f0g1, alice, bob)
########################################################################################################################
for port_id, upconv_freq, power_dbm in [
    (F0G1_PORT,  alice_f0g1_LO, alice_f0g1_power),
    (ALICE_PORT, alice_LO,      alice_power),
    (BOB_PORT,   alice_LO,      bob_power),
]:
    fsp, _ = get_full_scale_power_dBm_and_amplitude(power_dbm)
    port = machine.ports.get_mw_output(
        "con1", CAVITY_FEM_ID, port_id, create=True,
        upconverter_frequency=upconv_freq,
        band=get_band(upconv_freq),
        full_scale_power_dbm=fsp,
        delay=0,
        shareable=True,
    )
    port.upconverter_frequency = upconv_freq
    port.band                  = get_band(upconv_freq)
    port.full_scale_power_dbm  = fsp
    port.shareable             = True

########################################################################################################################
# %%  Resonator -- frequencies and power
########################################################################################################################
rr_fsp, rr_amp = get_full_scale_power_dBm_and_amplitude(readout_power)
for qubit in machine.qubits.values():
    qubit.resonator.f_01           = rr_freq
    qubit.resonator.RF_frequency   = rr_freq
    qubit.resonator.frequency_bare = rr_freq_bare
    qubit.resonator.time_of_flight = tof_ns
    qubit.resonator.smearing       = 0
    qubit.resonator.depletion_time = resonator_depletion_time_ns
    qubit.resonator.opx_output.upconverter_frequency = rr_LO
    qubit.resonator.opx_output.band                  = get_band(rr_LO)
    qubit.resonator.opx_output.full_scale_power_dbm  = rr_fsp
    qubit.resonator.opx_input.band                   = get_band(rr_LO)
    ro_op = qubit.resonator.operations.get("readout")
    if ro_op is not None:
        ro_op.amplitude = rr_amp
        ro_op.length    = readout_length_ns

########################################################################################################################
# %%  Qubit XY -- frequencies and pulses
########################################################################################################################
xy_fsp, xy_amp = get_full_scale_power_dBm_and_amplitude(drive_power)
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.f_01                                = xy_freq
    qubit.xy.RF_frequency                     = xy_freq
    qubit.xy.opx_output.upconverter_frequency = xy_LO
    qubit.xy.opx_output.band                  = get_band(xy_LO)
    qubit.xy.opx_output.full_scale_power_dbm  = xy_fsp
    qubit.T1                                  = T1
    qubit.anharmonicity                       = int(anharmonicity)
    qubit.grid_location                       = f"{k},0"
    qubit.thermalization_time_factor          = thermalization_time_factor
    qubit.sigma_time_factor                   = sigma_time_factor
    sat_op = qubit.xy.operations.get("saturation")
    if sat_op is not None:
        sat_op.amplitude      = 0.3
        sat_op.length         = saturation_length_ns
        sat_op.digital_marker = "ON"
    add_DragGaussian_pulses(qubit, xy_amp, x180_length_ns, gaussian_sigma_ns,
                            drag_alpha, drag_detuning, anharmonicity,
                            digital_marker="ON")

########################################################################################################################
# %%  EF and selective pulses
########################################################################################################################
for q_name, qubit in machine.qubits.items():
    xy = qubit.xy
    xy.operations["EF_x180"] = DragGaussianPulse(
        length="#../x180/length",
        amplitude=ef_x180_amplitude,
        sigma="#../x180/sigma",
        alpha=0.0,
        anharmonicity=int(anharmonicity),
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    xy.operations["EF_x90"] = DragGaussianPulse(
        length="#../EF_x180/length",
        amplitude=ef_x180_amplitude / 2,
        sigma="#../EF_x180/sigma",
        alpha="#../EF_x180/alpha",
        anharmonicity="#../EF_x180/anharmonicity",
        detuning="#../EF_x180/detuning",
        subtracted="#../EF_x180/subtracted",
        axis_angle=0,
        digital_marker="#../EF_x180/digital_marker",
    )
    sel_amp = xy_amp * (x180_length_ns / selective_x180_length_ns)
    xy.operations["selective_x180"] = DragGaussianPulse(
        length=selective_x180_length_ns,
        amplitude=sel_amp,
        sigma=selective_x180_length_ns // 5,
        alpha=0.0,
        anharmonicity=int(anharmonicity),
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )

########################################################################################################################
# %%  Cavity object (alice + bob)
########################################################################################################################
def _make_cavity_drive(port_id: int, freq: float, mode_id: str) -> XYDriveMW:
    return XYDriveMW(
        id=mode_id,
        opx_output=f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{port_id}",
        RF_frequency=freq,
    )

if CAVITY_ID not in machine.cavities:
    alice_mode = CavityMode(id="alice", cavity_mode_drive=_make_cavity_drive(ALICE_PORT, alice_freq, "alice_drive"))
    alice_mode.T1 = cavity_T1
    bob_mode = CavityMode(id="bob", cavity_mode_drive=_make_cavity_drive(BOB_PORT, bob_freq, "bob_drive"))
    bob_mode.T1 = cavity_T1
    machine.cavities[CAVITY_ID] = Cavity(id=CAVITY_ID, alice=alice_mode, bob=bob_mode)
    print(f"  Created cavity '{CAVITY_ID}' with alice and bob modes.")
else:
    for mode_name in ("alice", "bob"):
        mode = getattr(machine.cavities[CAVITY_ID], mode_name, None)
        if mode is not None:
            mode.T1 = cavity_T1

cav_fsp, cav_amp = get_full_scale_power_dBm_and_amplitude(alice_power)
for cav_name, cavity in machine.cavities.items():
    for mode_name, freq, port_id in (
        ("alice", alice_freq, ALICE_PORT),
        ("bob",   bob_freq,   BOB_PORT),
    ):
        mode = getattr(cavity, mode_name, None)
        if mode is None or mode.cavity_mode_drive is None:
            continue
        mode.cavity_mode_drive.id           = f"{mode_name}_drive"
        mode.cavity_mode_drive.opx_output   = f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{port_id}"
        mode.cavity_mode_drive.RF_frequency = freq
        mode.cavity_mode_drive.operations["saturation"] = SquarePulse(
            length=readout_length_ns, amplitude=cav_amp, digital_marker="ON")
        mode.cavity_mode_drive.operations["displacement"] = DragGaussianPulse(
            length=displacement_length_ns,
            amplitude=displacement_initial_amplitude_V,
            sigma=displacement_sigma_ns,
            alpha=0.0,
            anharmonicity=0,
            detuning=0.0,
            axis_angle=0,
            digital_marker="ON",
        )

CAVITY_THERM_FACTORS = {"alice": 3, "bob": 5}
for cav_name, cavity in machine.cavities.items():
    for mode_name, factor in CAVITY_THERM_FACTORS.items():
        mode = getattr(cavity, mode_name, None)
        if mode is not None:
            mode.thermalization_time_factor = factor

########################################################################################################################
# %%  CavityTransmonPair (with f0g1 sideband_drive)
########################################################################################################################
for q_name in machine.qubits:
    for mode_name in ("alice", "bob"):
        pair_key = f"{q_name}_{mode_name}"
        if pair_key not in machine.cavity_transmon_pairs:
            machine.cavity_transmon_pairs[pair_key] = CavityTransmonPair(
                qubit_name=q_name, cavity_mode_name=mode_name
            )
            print(f"  Created CavityTransmonPair '{pair_key}'")

    for pk in (f"{q_name}_alice", f"{q_name}_bob"):
        p = machine.cavity_transmon_pairs.get(pk)
        if p is not None and not hasattr(p, "parity_time"):
            p.parity_time = None

    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is not None and alice_pair.sideband_drive is None:
        f0g1_drive = XYDriveMW(
            id=f"{q_name}_alice_f0g1",
            opx_output=f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{F0G1_PORT}",
            RF_frequency=alice_f0g1_freq,
        )
        f0g1_drive.operations["saturation"] = SquarePulse(
            length=alice_f0g1_saturation_length_ns,
            amplitude=alice_f0g1_amp,
            digital_marker="ON",
        )
        f0g1_drive.operations["f0g1_pi"] = DragCosinePulse(
            length=alice_f0g1_pi_length_ns,
            amplitude=alice_f0g1_amp,
            alpha=0.0,
            anharmonicity=0,
            axis_angle=0.0,
            digital_marker="ON",
        )
        alice_pair.sideband_drive = f0g1_drive
        print(f"  Created sideband_drive for '{q_name}_alice' on FEM {CAVITY_FEM_ID}, port {F0G1_PORT}.")
    elif alice_pair is not None and alice_pair.sideband_drive is not None:
        alice_pair.sideband_drive.id          = f"{q_name}_alice_f0g1"
        alice_pair.sideband_drive.opx_output  = f"#/ports/mw_outputs/con1/{CAVITY_FEM_ID}/{F0G1_PORT}"
        alice_pair.sideband_drive.RF_frequency = alice_f0g1_freq
    # Note: bob sideband_drive stays None -- dedicated hardware not yet wired.

########################################################################################################################
# %%  Temporary calibration state
########################################################################################################################
if machine.temp_calibration is None:
    machine.temp_calibration = {}
for q_name in machine.qubits:
    if q_name not in machine.temp_calibration:
        machine.temp_calibration[q_name] = TemporaryCalibrationData(
            initial_resonator_f01=rr_freq,
            initial_resonator_RF_frequency=rr_freq,
        )
        print(f"  Initialized temp_calibration['{q_name}']")
    else:
        tc = machine.temp_calibration[q_name]
        tc.initial_resonator_f01          = rr_freq
        tc.initial_resonator_RF_frequency = rr_freq

########################################################################################################################
# %%  Save
########################################################################################################################
machine.save()
print("QUAM saved.")
with open("qua_config.json", "w+") as f:
    json.dump(machine.generate_config(), f, indent=4)
print("QUA config saved.")

## 3. Verify state

Sanity-check the saved QUAM structure after running sections 1 and 2.

In [ ]:
from quam_config import Quam

machine = Quam.load()

print("=== Qubits ===")
for q_name, qubit in machine.qubits.items():
    print(f"  {q_name}: f_01={qubit.f_01/1e9:.4f} GHz, T1={qubit.T1*1e6:.0f} us")
    print(f"    resonator: f={qubit.resonator.f_01/1e9:.4f} GHz, LO={qubit.resonator.opx_output.upconverter_frequency/1e9:.4f} GHz")
    print(f"    xy:        LO={qubit.xy.opx_output.upconverter_frequency/1e9:.4f} GHz")

print("\n=== Cavities ===")
for cav_name, cavity in machine.cavities.items():
    print(f"  {cav_name}:")
    for mode_name in ("alice", "bob"):
        mode = getattr(cavity, mode_name, None)
        if mode is not None and mode.cavity_mode_drive is not None:
            print(f"    {mode_name}: RF={mode.cavity_mode_drive.RF_frequency/1e9:.4f} GHz, T1={mode.T1*1e6:.0f} us")

print("\n=== CavityTransmonPairs ===")
for pair_key, pair in machine.cavity_transmon_pairs.items():
    sb = pair.sideband_drive
    sb_info = f"sideband_drive RF={sb.RF_frequency/1e9:.4f} GHz" if sb is not None else "no sideband_drive"
    print(f"  {pair_key}: {sb_info}")

print("\n=== MW-FEM ports (con1) ===")
for fem_slot in (1, 2):
    ports = machine.ports.mw_outputs.get("con1", {}).get(str(fem_slot), {})
    for port_id, port in ports.items():
        print(f"  FEM {fem_slot}, port {port_id}: LO={port.upconverter_frequency/1e9:.4f} GHz, "
              f"band={port.band}, fsp={port.full_scale_power_dbm} dBm")